# Understanding `search_similar` Function - Step by Step Tutorial

This notebook demonstrates how the `search_similar` function works in the RAG pipeline, including:
1. How embeddings are created
2. How they are stored in a vector database
3. How similarity search works
4. Complete end-to-end example

## Part 1: Overview of the Process

### The Complete Flow:

```
Text Document
     ↓
[1] Chunking (break into smaller pieces)
     ↓
[2] Generate Embeddings (text → vectors)
     ↓
[3] Store in Vector Database
     ↓
[4] Query: search_similar(query_vector, backend, top_k)
     ↓
[5] Get similar chunks ranked by score
```

In [2]:
# Import necessary modules
import sys
sys.path.append('..')

from vectordb.chunking import Chunk
from vectordb.embeddings import (
    EmbeddingConfig,
    get_provider,
    embed_chunks,
    create_embed_function
)
from vectordb.vector_store import (
    VectorStoreConfig,
    get_backend,
    store_embeddings,
    search_similar,
    SearchResult
)

## Part 2: Creating Sample Data (Chunks)

First, we create some sample text chunks. In a real scenario, these would come from your documents.

In [3]:
# Create sample chunks about different topics
sample_chunks = [
    Chunk(
        content="Python is a high-level programming language known for its simplicity and readability. It's widely used in data science and machine learning.",
        source_file="python_intro.md",
        chunk_index=0,
        start_char=0,
        end_char=150,
        metadata={"file_name": "python_intro.md", "topic": "programming"}
    ),
    Chunk(
        content="Machine learning is a subset of artificial intelligence that enables systems to learn from data without explicit programming. It uses algorithms to find patterns.",
        source_file="ml_basics.md",
        chunk_index=0,
        start_char=0,
        end_char=165,
        metadata={"file_name": "ml_basics.md", "topic": "AI"}
    ),
    Chunk(
        content="Vector databases store high-dimensional vectors and enable fast similarity search using techniques like cosine similarity and approximate nearest neighbors.",
        source_file="vector_db.md",
        chunk_index=0,
        start_char=0,
        end_char=155,
        metadata={"file_name": "vector_db.md", "topic": "databases"}
    ),
    Chunk(
        content="Cooking pasta requires boiling water with salt, adding pasta, and cooking for 8-12 minutes. Drain and serve with your favorite sauce.",
        source_file="cooking.md",
        chunk_index=0,
        start_char=0,
        end_char=130,
        metadata={"file_name": "cooking.md", "topic": "food"}
    ),
    Chunk(
        content="Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes that process information.",
        source_file="neural_nets.md",
        chunk_index=0,
        start_char=0,
        end_char=150,
        metadata={"file_name": "neural_nets.md", "topic": "AI"}
    )
]

print(f"Created {len(sample_chunks)} sample chunks")
for i, chunk in enumerate(sample_chunks):
    print(f"\nChunk {i}: {chunk.source_file}")
    print(f"  Content preview: {chunk.content[:80]}...")

Created 5 sample chunks

Chunk 0: python_intro.md
  Content preview: Python is a high-level programming language known for its simplicity and readabi...

Chunk 1: ml_basics.md
  Content preview: Machine learning is a subset of artificial intelligence that enables systems to ...

Chunk 2: vector_db.md
  Content preview: Vector databases store high-dimensional vectors and enable fast similarity searc...

Chunk 3: cooking.md
  Content preview: Cooking pasta requires boiling water with salt, adding pasta, and cooking for 8-...

Chunk 4: neural_nets.md
  Content preview: Neural networks are computing systems inspired by biological neural networks. Th...


## Part 3: Understanding Embeddings

### What are embeddings?
Embeddings are numerical representations (vectors) of text. Similar texts have similar vectors.

### How it works:
1. **Model**: Uses a pre-trained model (e.g., 'all-MiniLM-L6-v2')
2. **Input**: Text string
3. **Output**: Array of numbers (e.g., 384 dimensions)

Example:
- "Python programming" → [0.23, -0.14, 0.56, ..., 0.12] (384 numbers)
- "Coding in Python" → [0.21, -0.15, 0.54, ..., 0.11] (similar vector!)
- "Cooking pasta" → [-0.45, 0.32, -0.12, ..., 0.67] (different vector!)

In [4]:
# Step 1: Create an embedding provider (using local model, free!)
embedding_config = EmbeddingConfig(
    model="all-MiniLM-L6-v2",  # Fast, 384-dimensional embeddings
    batch_size=10,
    normalize=True
)

print("Creating embedding provider...")
provider = get_provider("sentence-transformers", embedding_config)

print(f"Model: {provider.model_name}")
print(f"Dimensions: {provider.dimensions}")
print("\nThis model will convert each text into a {}-dimensional vector".format(provider.dimensions))

Creating embedding provider...
Model: all-MiniLM-L6-v2
Dimensions: 384

This model will convert each text into a 384-dimensional vector


In [5]:
# Step 2: Generate embeddings for our chunks
print("Generating embeddings...\n")
embeddings = embed_chunks(sample_chunks, provider, batch_size=10)

print(f"Generated {len(embeddings)} embeddings\n")

# Let's examine one embedding
first_emb = embeddings[0]
print(f"Embedding for: '{first_emb.chunk.content[:60]}...'")
print(f"\nVector (first 10 dimensions): {first_emb.vector[:10]}")
print(f"Total dimensions: {first_emb.dimensions}")
print(f"Model used: {first_emb.model}")

Generating embeddings...

Generated 5 embeddings

Embedding for: 'Python is a high-level programming language known for its si...'

Vector (first 10 dimensions): (-0.05176158249378204, 0.004511675331741571, -0.03879000246524811, 0.04376062750816345, -0.020291123539209366, -0.1404152661561966, -0.004258740693330765, 0.056598734110593796, -0.08964025229215622, -0.010350902564823627)
Total dimensions: 384
Model used: all-MiniLM-L6-v2


## Part 4: Storing Embeddings in Vector Database

We'll use the **simple** backend (JSON file-based) for this tutorial.
In production, you'd use PgVector for PostgreSQL.

In [6]:
# Create vector store configuration
vector_config = VectorStoreConfig(
    collection_name="demo_collection",
    persist_directory="../data/demo_vectordb",
    distance_metric="cosine"  # Cosine similarity (most common for text)
)

# Get backend (simple JSON-based for demo)
print("Creating vector store backend...")
backend = get_backend("simple", vector_config)
print("Backend created!\n")

# Store embeddings
print("Storing embeddings in vector database...")
embedding_ids = store_embeddings(embeddings, backend)

print(f"\nStored {len(embedding_ids)} embeddings")
print(f"\nEmbedding IDs:")
for id_ in embedding_ids:
    print(f"  - {id_}")

Creating vector store backend...
Backend created!

Storing embeddings in vector database...

Stored 5 embeddings

Embedding IDs:
  - python_intro.md_0
  - ml_basics.md_0
  - vector_db.md_0
  - cooking.md_0
  - neural_nets.md_0


## Part 5: Understanding `search_similar` Function

### Function Signature:
```python
def search_similar(
    query_vector: list[float],  # Your query as a vector
    backend: VectorStoreBackend, # The database
    top_k: int = 10             # How many results to return
) -> list[SearchResult]:
```

### Step-by-Step Process:

1. **Input**: Takes a query vector (not text!)
2. **Backend Search**: Calls `backend.search(query_vector, top_k)`
3. **Backend computes**: Similarity scores between query and all stored vectors
4. **Ranking**: Sorts by score (highest = most similar)
5. **Return top_k**: Returns the top K most similar chunks
6. **Output**: List of SearchResult objects with:
   - `chunk`: The original text chunk
   - `score`: Similarity score (0-1, higher = more similar)
   - `embedding_id`: Unique identifier

### How Similarity is Computed:

**Cosine Similarity** (most common):
```
similarity = (vector_A · vector_B) / (|vector_A| × |vector_B|)
```

Range: -1 to 1 (usually 0 to 1 for normalized vectors)
- 1.0 = identical
- 0.5 = somewhat similar
- 0.0 = orthogonal (unrelated)

In [7]:
# Let's look at the actual code of search_similar
import inspect
from vectordb.vector_store import search_similar

print("SOURCE CODE of search_similar:\n")
print(inspect.getsource(search_similar))

SOURCE CODE of search_similar:

def search_similar(
    query_vector: list[float], backend: VectorStoreBackend, top_k: int = 10
) -> list[SearchResult]:
    """
    Search for similar embeddings.

    Args:
        query_vector: Query embedding vector
        backend: Vector store backend
        top_k: Number of results to return

    Returns:
        List of SearchResult objects
    """
    results = backend.search(query_vector, top_k)

    search_results = []
    for emb_id, score, metadata in results:
        chunk = Chunk(
            content=metadata.get("content", metadata.get("document", "")),
            source_file=metadata.get("source_file", ""),
            chunk_index=metadata.get("chunk_index", 0),
            start_char=metadata.get("start_char", 0),
            end_char=metadata.get("end_char", 0),
            metadata={"file_name": metadata.get("file_name", "")},
        )
        search_results.append(
            SearchResult(chunk=chunk, score=score, embedding_id=em

## Part 6: Searching for Similar Content

Now let's perform actual searches!

In [8]:
# Create an embedding function for queries
embed_fn = create_embed_function("sentence-transformers", embedding_config)

# Test queries
queries = [
    "How does artificial intelligence work?",
    "Programming languages for data science",
    "How to make dinner?"
]

for query_text in queries:
    print("="*80)
    print(f"QUERY: {query_text}")
    print("="*80)
    
    # STEP 1: Convert query text to vector
    print("\n[Step 1] Converting query to embedding vector...")
    query_vector = embed_fn(query_text)
    print(f"  Query vector (first 10 dims): {query_vector[:10]}")
    print(f"  Total dimensions: {len(query_vector)}")
    
    # STEP 2: Search for similar vectors
    print("\n[Step 2] Searching for similar content...")
    results = search_similar(query_vector, backend, top_k=3)
    
    # STEP 3: Display results
    print(f"\n[Step 3] Found {len(results)} results:\n")
    
    for i, result in enumerate(results, 1):
        print(f"Rank {i}:")
        print(f"  Score: {result.score:.4f} (1.0 = perfect match)")
        print(f"  Source: {result.chunk.source_file}")
        print(f"  Content: {result.chunk.content[:100]}...")
        print(f"  ID: {result.embedding_id}")
        print()
    
    print("\n")

QUERY: How does artificial intelligence work?

[Step 1] Converting query to embedding vector...
  Query vector (first 10 dims): [0.03628707677125931, -0.016797125339508057, 0.021820450201630592, -0.0134316710755229, -0.003979697357863188, -0.023136818781495094, 0.047955580055713654, -0.0063441116362810135, 0.004620127379894257, 0.06676146388053894]
  Total dimensions: 384

[Step 2] Searching for similar content...

[Step 3] Found 3 results:

Rank 1:
  Score: 0.5684 (1.0 = perfect match)
  Source: ml_basics.md
  Content: Machine learning is a subset of artificial intelligence that enables systems to learn from data with...
  ID: ml_basics.md_0

Rank 2:
  Score: 0.4301 (1.0 = perfect match)
  Source: neural_nets.md
  Content: Neural networks are computing systems inspired by biological neural networks. They consist of layers...
  ID: neural_nets.md_0

Rank 3:
  Score: 0.2678 (1.0 = perfect match)
  Source: python_intro.md
  Content: Python is a high-level programming language known for i

## Part 7: Understanding the Backend Search

Let's look at what happens inside `backend.search()` for the simple backend:

In [9]:
# Recreate the simple backend's search logic to understand it

def cosine_similarity_demo(v1: list[float], v2: list[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    # Dot product: sum of element-wise multiplication
    dot = sum(a * b for a, b in zip(v1, v2))
    
    # Magnitude of v1: square root of sum of squares
    norm1 = sum(a * a for a in v1) ** 0.5
    
    # Magnitude of v2
    norm2 = sum(b * b for b in v2) ** 0.5
    
    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0
    
    # Cosine similarity formula
    return dot / (norm1 * norm2)

# Demonstrate with actual vectors
query = "machine learning"
query_vector = embed_fn(query)

print(f"Query: '{query}'\n")
print("Computing similarity scores manually:\n")

for emb in embeddings:
    score = cosine_similarity_demo(query_vector, list(emb.vector))
    print(f"Score: {score:.4f} | {emb.chunk.source_file}")
    print(f"  Content: {emb.chunk.content[:70]}...\n")

Query: 'machine learning'

Computing similarity scores manually:

Score: 0.1784 | python_intro.md
  Content: Python is a high-level programming language known for its simplicity a...

Score: 0.6298 | ml_basics.md
  Content: Machine learning is a subset of artificial intelligence that enables s...

Score: 0.2800 | vector_db.md
  Content: Vector databases store high-dimensional vectors and enable fast simila...

Score: 0.0285 | cooking.md
  Content: Cooking pasta requires boiling water with salt, adding pasta, and cook...

Score: 0.2695 | neural_nets.md
  Content: Neural networks are computing systems inspired by biological neural ne...



## Part 8: Where is `search_similar` Called?

The `search_similar` function is called in two main places:

### 1. Direct Usage:
```python
from vectordb.vector_store import search_similar

results = search_similar(query_vector, backend, top_k=10)
```

### 2. Via `query_vector_store` (convenience wrapper):
```python
from vectordb.vector_store import query_vector_store

# This function:
# 1. Converts query_text to vector using embed_fn
# 2. Calls search_similar with that vector
results = query_vector_store(
    query_text="What is Python?",
    backend=backend,
    embed_fn=embed_fn,
    top_k=10
)
```

Let's demonstrate both:

In [10]:
from vectordb.vector_store import query_vector_store

query_text = "Tell me about Python programming"

# METHOD 1: Manual (2 steps)
print("METHOD 1: Using search_similar (manual)")
print("-" * 50)
query_vector = embed_fn(query_text)
results_method1 = search_similar(query_vector, backend, top_k=2)

for r in results_method1:
    print(f"Score: {r.score:.4f} | {r.chunk.source_file}")

print("\n")

# METHOD 2: Using convenience wrapper (1 step)
print("METHOD 2: Using query_vector_store (wrapper)")
print("-" * 50)
results_method2 = query_vector_store(query_text, backend, embed_fn, top_k=2)

for r in results_method2:
    print(f"Score: {r.score:.4f} | {r.chunk.source_file}")

print("\n✅ Both methods produce identical results!")

METHOD 1: Using search_similar (manual)
--------------------------------------------------
Score: 0.8125 | python_intro.md
Score: 0.2554 | ml_basics.md


METHOD 2: Using query_vector_store (wrapper)
--------------------------------------------------
Score: 0.8125 | python_intro.md
Score: 0.2554 | ml_basics.md

✅ Both methods produce identical results!


## Part 9: Complete Flow Diagram

```
USER QUERY: "What is machine learning?"
        |
        v
[1] embed_fn(query_text)
        |
        v
    query_vector: [0.12, -0.45, 0.67, ...] (384 numbers)
        |
        v
[2] search_similar(query_vector, backend, top_k=5)
        |
        v
[3] backend.search(query_vector, top_k)
        |
        +-- For each stored embedding:
        |       |
        |       v
        |   Calculate: cosine_similarity(query_vector, stored_vector)
        |       |
        |       v
        |   Score: 0.0 to 1.0
        |
        v
[4] Sort all results by score (descending)
        |
        v
[5] Return top K results
        |
        v
    [
        SearchResult(chunk, score=0.85, id),
        SearchResult(chunk, score=0.72, id),
        SearchResult(chunk, score=0.68, id),
        ...
    ]
```

## Part 10: Key Concepts Summary

### 1. **Embeddings** (Text → Numbers)
- Convert text to vectors using ML models
- Similar text = similar vectors
- Example: "Python coding" and "Programming in Python" have similar embeddings

### 2. **Vector Database** (Storage)
- Stores embeddings with metadata
- Enables fast similarity search
- Backends: Simple (JSON), PgVector (PostgreSQL)

### 3. **Similarity Search** (Finding Related Content)
- Compare query vector to all stored vectors
- Use distance metrics (cosine, L2, etc.)
- Return top K most similar results

### 4. **search_similar Function**
```python
def search_similar(
    query_vector,  # Already converted to vector!
    backend,       # Database connection
    top_k          # Number of results
):
    # 1. Ask backend to search
    results = backend.search(query_vector, top_k)
    
    # 2. Convert raw results to SearchResult objects
    search_results = []
    for id, score, metadata in results:
        chunk = Chunk(...)  # Reconstruct chunk from metadata
        search_results.append(SearchResult(chunk, score, id))
    
    # 3. Return ranked results
    return search_results
```

### 5. **When to Use**
- **RAG (Retrieval Augmented Generation)**: Find relevant context for LLMs
- **Semantic Search**: Search by meaning, not just keywords
- **Recommendation Systems**: Find similar items
- **Duplicate Detection**: Find similar documents

## Part 11: Practical Exercise

Try modifying the code below to:
1. Add your own chunks
2. Search with different queries
3. Experiment with different `top_k` values

In [ ]:
# YOUR TURN: Add more chunks and search!

# Add new chunks here
new_chunks = [
    # TODO: Add your own chunks
]

# Generate embeddings for new chunks
if new_chunks:
    new_embeddings = embed_chunks(new_chunks, provider)
    store_embeddings(new_embeddings, backend)
    print(f"Added {len(new_embeddings)} new embeddings")

# Try your own query
my_query = "YOUR QUERY HERE"
results = query_vector_store(my_query, backend, embed_fn, top_k=3)

print(f"\nResults for: '{my_query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [{result.score:.4f}] {result.chunk.content[:80]}...")

## Conclusion

You now understand:

✅ How embeddings convert text to vectors  
✅ How vector databases store and index embeddings  
✅ How `search_similar` finds relevant content  
✅ How similarity is calculated (cosine similarity)  
✅ Where and how `search_similar` is used  
✅ The complete end-to-end RAG retrieval process  

### Next Steps:
- Try different embedding models (see `embeddings.py`)
- Experiment with PgVector backend for production
- Integrate with LLMs for complete RAG pipeline
- Build your own semantic search application!